# ASR Evaluation

Use Meta's [`Llama-Guard-3-8b`](https://huggingface.co/meta-llama/Llama-Guard-3-8b) to evaluate the safety of a model.

Prompts from 100 samples of the `AdvBench` dataset. 

In [1]:
import unsloth
from unsloth import FastLanguageModel
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import pandas as pd
from tqdm import tqdm
import gc
import os

from utils import get_prompt, get_guard_prompt, get_guard_score

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/ajl64/miniconda3/envs/LoX/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!


Unable to import `torchao` Tensor objects. This may affect loading checkpoints serialized with `torchao`


In [2]:
# ===== CONFIG =====
MODEL          = "./models/LoX-10-10" # model to evaluate
MODEL_NAME     = os.path.split(MODEL)[1]
JUDGE_MODEL    = "meta-llama/Llama-Guard-3-8b"

ADVBENCH_PATH  = "../data/harmful_behaviors.csv"
SAVE_DIR       = "./output/"
SAVE_PATH      = os.path.join(
    SAVE_DIR, 
    "ASR-" + MODEL_NAME + ".csv"
)

SEED           = abs(hash("chud"))
BATCH_SIZE     = 1
N              = 100 # test w/ 100 samples as in LoX paper

torch.manual_seed(SEED)
print(
    f"Evaluating {MODEL_NAME} (at {MODEL})\n"
    f"\t- With judge {JUDGE_MODEL}\n"
    f"\t- With {N} sample(s) from {ADVBENCH_PATH}\n"
    f"\t- Saving to {SAVE_PATH}"
)

Evaluating LoX-10-10 (at ./models/LoX-10-10)
	- With judge meta-llama/Llama-Guard-3-8b
	- With 100 sample(s) from ../data/harmful_behaviors.csv
	- Saving to ./output/ASR-LoX-10-10.csv


In [3]:
# ===== Inference (Completions) =====
def generate_completions(samples):
    model, tokenizer = FastLanguageModel.from_pretrained(
        MODEL, low_cpu_mem_usage=True,
    )
    FastLanguageModel.for_inference(model)
    model.eval()

    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    prompts = [get_prompt(instruction)[0] for instruction in samples]
    instructions, completions = [], []

    for i in tqdm(range(0, len(prompts), BATCH_SIZE)):
        batch_prompts = prompts[i:i + BATCH_SIZE]
        inputs = tokenizer(
            batch_prompts,
            return_tensors='pt',
            padding=True
        ).to(model.device)

        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                top_k=50,
                repetition_penalty=1.2
            )

        for j, seq in enumerate(output):
            input_len = inputs['input_ids'][j].shape[0]
            text = tokenizer.decode(seq[input_len:], skip_special_tokens=True)
            
            instructions.append(samples[i+j])
            completions.append(text.strip() if text.strip() else "[EMPTY ANSWER]")

    del model
    gc.collect()

    return instructions, completions

In [4]:
# ===== Inference (Judging) =====
def judge_completions(instructions, completions):
    model, _ = FastLanguageModel.from_pretrained(
        JUDGE_MODEL,
        low_cpu_mem_usage=True,
        device_map="auto",
    )
    FastLanguageModel.for_inference(model)
    
    tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL, padding_side='left')
    tokenizer.pad_token = tokenizer.eos_token

    all_prompts = [
        tokenizer.apply_chat_template(
            get_guard_prompt(inst, comp),
            tokenize=False, add_generation_prompt=True
        )
        for inst, comp in zip(instructions, completions)
    ]

    scores, judge_outputs = [], []
    for i in tqdm(range(0, len(all_prompts), BATCH_SIZE), total=(len(all_prompts) + BATCH_SIZE - 1) // BATCH_SIZE):
        batch_prompts = all_prompts[i:i+BATCH_SIZE]
        inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
        ).to(model.device)

        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=False
            )

        for j, output_seq in enumerate(output):
            input_len = inputs["input_ids"][j].shape[0]
            new_tokens = output_seq[input_len:]
            judge_out = tokenizer.decode(
                new_tokens,
                skip_special_tokens=True
            ).strip()
            
            score = get_guard_score(judge_out)
            if score not in (0, 1):
                print(f"Judge failed on input {j}")
                print("\n===== JUDGE OUTPUT =====\n")
                print(judge_out)
                print("\n")
            else:
                scores.append(score)
                judge_outputs.append(judge_out)

    del model
    gc.collect()

    return scores, judge_outputs


In [5]:
# ===== Load Data =====
advbench_samples = list(pd.read_csv(ADVBENCH_PATH)["goal"])[:N]

# ===== Load Models =====
judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL, padding_side='left')
model_tokenizer = AutoTokenizer.from_pretrained(MODEL)

print("===== Pass 1: Generating Samples =====")
instructions, completions = generate_completions(advbench_samples)

print("===== Pass 2: Judging Respones =====")
scores, judge_outputs = judge_completions(instructions, completions)

# ===== Results =====
attack_successes = sum(scores)
total_samples = len(scores)
asr_percent = 100 * (attack_successes / total_samples)
print(f"Evaluation: {attack_successes} / {total_samples} attacks succeeded.")
print(f"ASR: {asr_percent:.3f}%")

pd.DataFrame({
        "instruction":  instructions,
        "completion":   completions,
        "score":        scores,
        "judge_output": judge_outputs,
    }).to_csv(SAVE_PATH, index=False)

print(f"Results saved to {SAVE_PATH}")

===== Pass 1: Generating Samples =====
==((====))==  Unsloth 2025.11.1: Fast Llama patching. Transformers: 4.57.2.
   \\   /|    NVIDIA GeForce RTX 4080 Laptop GPU. Num GPUs = 1. Max memory: 11.994 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]/home/ajl64/miniconda3/envs/LoX/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading checkpoint shards: 100%|██████████| 3/3 [00:21<00:00,  7.05s/it]
Unsloth 2025.11.1 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.
100%|██████████| 100/100 [07:11<00:00,  4.31s/it]


===== Pass 2: Judging Respones =====
==((====))==  Unsloth 2025.11.1: Fast Llama patching. Transformers: 4.57.2.
   \\   /|    NVIDIA GeForce RTX 4080 Laptop GPU. Num GPUs = 1. Max memory: 11.994 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 8.9. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]/home/ajl64/miniconda3/envs/LoX/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading checkpoint shards: 100%|██████████| 4/4 [00:21<00:00,  5.46s/it]
Unsloth: We successfully patched the tokenizer to add a {% if add_generation_prompt %} to the chat_template.
This is not a bug, but please notify the maintainers of `meta-llama/Llama-Guard-3-8b` - thanks!


meta-llama/Llama-Guard-3-8b does not have a padding token! Will use pad_token = <|finetune_right_pad_id|>.


100%|██████████| 100/100 [00:33<00:00,  3.02it/s]

Evaluation: 4 / 100 attacks succeeded.
ASR: 4.000%
Results saved to ./output/ASR-LoX-10-10.csv
